# 📁 UJI MODEL AI SPRAYBOT — KHUSUS 1 FOLDER (BATCH)
### Pengujian Massal Model YOLOv8s-Seg Terbaru (`weights/spraybot_seg_video_finetuned.pt`)
**Proyek AI Spraybot — MBKM Riset Magang Industri PT Paragon Technology and Innovation**

Notebook ini dibuat **khusus untuk menguji seluruh gambar dalam 1 folder pilihan Anda**.  
Fitur utama:
1. Memproses banyak citra sekaligus secara otomatis dengan *real-time progress bar*.
2. Menyimpan setiap citra hasil prediksi dengan masker hijau transparan di `testing/output/batch_overlays/`.
3. Menghitung statistik ringkasan semprotan (Jangkauan rata-rata, lebar rata-rata, luas plume, dan konsistensi).
4. Mengekspor seluruh metrik ke format **Laporan Excel / CSV** (`testing/output/laporan_uji_folder.csv`).
5. Menampilkan **Galeri Visualisasi Grid** hasil deteksi.

## 1. Setup Environment & Memuat Model AI Terbaru

In [ ]:
from pathlib import Path
import os, sys, cv2, torch, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Konfigurasi Direktori Kerja Cerdas
CWD = Path.cwd().resolve()
if CWD.name == "scripts" and CWD.parent.name == "testing":
    TESTING_DIR = CWD.parent
    ROOT_DIR = TESTING_DIR.parent
elif CWD.name == "testing":
    TESTING_DIR = CWD
    ROOT_DIR = TESTING_DIR.parent
else:
    ROOT_DIR = CWD
    TESTING_DIR = ROOT_DIR / "testing"

OUTPUT_DIR = TESTING_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Path Model Bobot Terbaru (Prioritas: Model 122 video terbaru spraybot_seg_finetuned.pt)
candidate_models = [
    TESTING_DIR / "weights" / "spraybot_seg_finetuned.pt",
    ROOT_DIR / "weights" / "spraybot_seg_finetuned.pt",
    TESTING_DIR / "testing_18-11-2026" / "weights" / "spraybot_seg_finetuned.pt",
    ROOT_DIR / "weights" / "spraybot_seg_video_finetuned.pt",
]

MODEL_PATH = None
for cm in candidate_models:
    if cm.exists():
        MODEL_PATH = cm
        break

if MODEL_PATH is None:
    raise FileNotFoundError("Model bobot spraybot_seg_finetuned.pt tidak ditemukan!")

print("=" * 70)
print(f"CUDA Tersedia       : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Perangkat Komputasi : {torch.cuda.get_device_name(0)}")
print(f"Model Digunakan     : {MODEL_PATH.name}")
print(f"Path Model          : {MODEL_PATH}")
print(f"Folder Output Hasil : {OUTPUT_DIR}")
print("=" * 70)

# Load Model
model = YOLO(str(MODEL_PATH))
print("Model AI Spraybot terbaru berhasil dimuat dan siap digunakan!")


## 2. Modul Pemrosesan Geometri Fisik Massal

In [ ]:
def calculate_spray_geometry(pts, img_shape, px_per_cm=20.46):
    """Menghitung parameter geometri fisik semprotan aerosol."""
    h, w = img_shape[:2]
    pts = np.asarray(pts, dtype=np.float32)
    
    min_x_idx = np.argmin(pts[:, 0])
    nozzle_pt = (int(pts[min_x_idx, 0]), int(pts[min_x_idx, 1]))
    
    max_x = np.max(pts[:, 0])
    reach_px = float(max_x - nozzle_pt[0])
    reach_cm = reach_px / px_per_cm
    
    min_y = np.min(pts[:, 1])
    max_y = np.max(pts[:, 1])
    width_px = float(max_y - min_y)
    width_cm = width_px / px_per_cm
    
    area_px2 = float(cv2.contourArea(pts.astype(np.int32)))
    area_cm2 = area_px2 / (px_per_cm ** 2)
    
    mid_x = nozzle_pt[0] + (reach_px * 0.5)
    near_mid = pts[np.abs(pts[:, 0] - mid_x) < (reach_px * 0.15)]
    if len(near_mid) >= 2:
        top_y = np.min(near_mid[:, 1])
        bot_y = np.max(near_mid[:, 1])
        angle_rad = 2 * np.arctan2((bot_y - top_y) / 2.0, (mid_x - nozzle_pt[0]))
        cone_angle_deg = float(np.degrees(angle_rad))
    else:
        cone_angle_deg = float(np.degrees(2 * np.arctan2(width_px / 2.0, max(1.0, reach_px))))
        
    return {
        "nozzle": nozzle_pt,
        "reach_px": round(reach_px, 1),
        "reach_cm": round(reach_cm, 2),
        "width_px": round(width_px, 1),
        "width_cm": round(width_cm, 2),
        "area_cm2": round(area_cm2, 2),
        "cone_angle_deg": round(cone_angle_deg, 1)
    }

print("Modul geometri batch siap digunakan!")

## 3. Eksekusi Uji 1 Folder (Pilih Path Folder Anda)
Silakan ganti nilai variabel `PATH_FOLDER_PILIHAN` di bawah ini dengan folder berisi gambar yang ingin Anda uji:

In [ ]:
# ==============================================================================
# 🎯 MASUKKAN PATH FOLDER ANDA DI BAWAH INI:
# ==============================================================================
PATH_FOLDER_PILIHAN = "dataset/dataset_video_frames_cropped/C0352-Normal-Kahf-1"

# Batas Maksimal Gambar yang Diuji (Isi misal 15, 30, atau None untuk SEMUA gambar)
BATAS_MAKSIMAL_GAMBAR = 15

# Ambang Batas Confidence
CONFIDENCE_THRESHOLD = 0.15

# Skala Kalibrasi px/cm
SKALA_PX_PER_CM = 20.46
# ==============================================================================

folder_p = Path(PATH_FOLDER_PILIHAN)
if not folder_p.is_absolute():
    folder_p = (ROOT_DIR / folder_p).resolve()

if not folder_p.exists():
    raise FileNotFoundError(f"Folder tidak ditemukan: {folder_p}")

all_imgs = sorted(list(folder_p.glob("*.jpg")) + list(folder_p.glob("*.png")))
if BATAS_MAKSIMAL_GAMBAR is not None:
    img_files = all_imgs[:BATAS_MAKSIMAL_GAMBAR]
else:
    img_files = all_imgs

print(f"Menjalankan analisis batch pada folder : {folder_p.name}")
print(f"Jumlah citra yang akan diproses       : {len(img_files)} citra (dari total {len(all_imgs)})
")

batch_results = []
saved_overlay_paths = []
t_start_batch = time.time()

for i, img_p in enumerate(img_files, 1):
    img_bgr = cv2.imread(str(img_p))
    if img_bgr is None:
        continue
    h, w = img_bgr.shape[:2]
    
    t0 = time.time()
    res = model.predict(source=img_bgr, conf=CONFIDENCE_THRESHOLD, device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
    dt = (time.time() - t0) * 1000.0
    
    detected = res.masks is not None and len(res.masks) > 0
    row = {
        "No": i,
        "File": img_p.name,
        "Status": "VALID_PLUME" if detected else "NEGATIVE_SAMPLE",
        "Conf (%)": 0.0,
        "Reach (cm)": 0.0,
        "Reach (px)": 0.0,
        "Width (cm)": 0.0,
        "Width (px)": 0.0,
        "Area (cm2)": 0.0,
        "Angle (deg)": 0.0,
        "Nozzle": "(0, 0)",
        "Infer Time (ms)": round(dt, 1)
    }
    
    vis = img_bgr.copy()
    if detected:
        best_idx = int(np.argmax(res.boxes.conf.cpu().numpy()))
        best_conf = float(res.boxes.conf[best_idx].cpu().numpy())
        pts = res.masks.xy[best_idx]
        geo = calculate_spray_geometry(pts, img_bgr.shape, px_per_cm=SKALA_PX_PER_CM)
        
        row.update({
            "Conf (%)": round(best_conf * 100, 1),
            "Reach (cm)": geo["reach_cm"],
            "Reach (px)": geo["reach_px"],
            "Width (cm)": geo["width_cm"],
            "Width (px)": geo["width_px"],
            "Area (cm2)": geo["area_cm2"],
            "Angle (deg)": geo["cone_angle_deg"],
            "Nozzle": f"({geo['nozzle'][0]}, {geo['nozzle'][1]})"
        })
        
        # Buat overlay
        overlay = np.zeros_like(img_bgr)
        pts_int = pts.astype(np.int32).reshape((-1, 1, 2))
        cv2.fillPoly(overlay, [pts_int], (0, 255, 0))
        vis = cv2.addWeighted(vis, 1.0, overlay, 0.45, 0)
        cv2.polylines(vis, [pts_int], True, (0, 255, 0), 2)
        
        # Bounding box & Nozzle
        box = res.boxes.xyxy[best_idx].cpu().numpy().astype(int)
        cv2.rectangle(vis, (box[0], box[1]), (box[2], box[3]), (0, 255, 255), 2)
        nx, ny = geo["nozzle"]
        cv2.circle(vis, (nx, ny), 7, (0, 0, 255), -1)
        cv2.putText(vis, f"Nozzle ({nx},{ny})", (max(10, nx - 40), max(25, ny - 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 2)
        
    out_overlay_file = OVERLAYS_DIR / f"pred_{img_p.stem}.jpg"
    cv2.imwrite(str(out_overlay_file), vis, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    saved_overlay_paths.append(out_overlay_file)
    batch_results.append(row)
    
    stat_icon = "🟢" if detected else "⚪"
    print(f"  [{i:02d}/{len(img_files):02d}] {stat_icon} {img_p.name:<40} -> Reach: {row['Reach (cm)']:>5.1f} cm | Conf: {row['Conf (%)']:>5.1f}% ({row['Infer Time (ms)']} ms)")

total_batch_time = time.time() - t_start_batch
df_batch = pd.DataFrame(batch_results)

# Simpan Laporan CSV
csv_export_path = OUTPUT_DIR / f"laporan_uji_folder_{folder_p.name}.csv"
df_batch.to_csv(csv_export_path, index=False)

print("
" + "=" * 70)
print(f"PENGUJIAN BATCH FOLDER SELESAI DALAM {total_batch_time:.2f} DETIK!")
print(f"Total Citra Diuji        : {len(batch_results)}")
valid_count = sum(1 for r in batch_results if r['Status'] == 'VALID_PLUME')
print(f"Semprotan Terdeteksi     : {valid_count} ({valid_count/len(batch_results):.1%})")
print(f"Laporan CSV Tersimpan di : {csv_export_path}")
print(f"Galeri Overlay Disimpan di: {OVERLAYS_DIR}")
print("=" * 70)

# Tampilkan Tabel Hasil
display(df_batch)

## 4. Galeri Visualisasi Hasil Deteksi Batch (Grid)
Menampilkan cuplikan galeri visualisasi dari frame-frame yang telah diproses di folder tersebut:

In [ ]:
# Tampilkan Galeri 6 Sampel Pertama
sample_overlays = saved_overlay_paths[:6]
if sample_overlays:
    n_show = len(sample_overlays)
    n_cols = 2
    n_rows = (n_show + 1) // 2
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4.5 * n_rows), facecolor="white")
    axes = np.array(axes).reshape(-1)
    
    for idx, ov_p in enumerate(sample_overlays):
        im = Image.open(ov_p)
        axes[idx].imshow(im)
        axes[idx].set_title(f"Hasil AI: {ov_p.name.replace('pred_', '')}", fontsize=10, fontweight="bold", pad=6)
        axes[idx].axis("off")
        
    for idx in range(n_show, len(axes)):
        axes[idx].axis("off")
        
    plt.tight_layout()
    gallery_path = OUTPUT_DIR / f"galeri_grid_{folder_p.name}.png"
    fig.savefig(gallery_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Galeri visualisasi tersimpan di: {gallery_path}")